# 03 EDA and Univariate Analysis

**Purpose**

This notebook performs exploratory data analysis and univariate risk analysis on the modelling population prepared in Notebook 02.

**Important convention**

`observed_bad_rate` is an empirical historical charge-off rate calculated as:

`bad_count / total_accounts`

It is **not** a modelled PD.

**Inputs**

`../data/processed/02.target_definition_and_leakage/initial_modeling_dataset.pkl`


`target_bad = 1 for Charged Off, 0 for Fully Paid`

**Outputs**

Results are saved under:

`../data/processed/03.eda_and_univariate_analysis/`

as both Excel and pickle artifacts.

In [ ]:
import os
import pickle
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

## 1. Paths and configuration

In [ ]:
NOTEBOOK_NAME = '03_eda_and_univariate_analysis'

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]

def find_project_root():
    for root in PROJECT_ROOT_CANDIDATES:
        if (root / 'data').exists() or (root / 'loan_processed_data.csv').exists():
            return root
    return Path.cwd()

PROJECT_ROOT = find_project_root()

RAW_DATA_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'raw' / 'loan_processed_data.csv',
    PROJECT_ROOT / 'loan_processed_data.csv',
    Path('/mnt/data/loan_processed_data.csv')
]

NB02_DIR_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed' / '02.target_definition_and_leakage',
    Path.cwd() / '..' / 'data' / 'processed' / '02.target_definition_and_leakage',
    Path('/mnt/data/data/processed/02.target_definition_and_leakage')
]

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / '03.eda_and_univariate_analysis'
CHART_DIR = OUTPUT_DIR / 'charts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_OUTPUT_PATH = OUTPUT_DIR / 'eda_and_univariate_analysis_report.xlsx'
PKL_OUTPUT_PATH = OUTPUT_DIR / 'eda_and_univariate_analysis_artifacts.pkl'
EDA_DATASET_PATH = OUTPUT_DIR / 'eda_modeling_dataset.pkl'

TARGET_COL = 'target_bad'
ORIGINAL_TARGET_COL = 'loan_status'
BAD_LABEL = 'Charged Off'
GOOD_LABEL = 'Fully Paid'

EXCLUDED_ISSUE_QUARTERS = ['2018Q3', '2018Q4']

MAX_CATEGORICAL_LEVELS = 50
N_NUMERIC_BINS = 10
MIN_BIN_SIZE = 100

print('Project root:', PROJECT_ROOT.resolve())
print('Output directory:', OUTPUT_DIR.resolve())

## 2. Load modelling population

Notebook 03 should use the modelling population from Notebook 02. That dataset already has:

- maturity-bias periods excluded: `2018Q3`, `2018Q4`
- leakage variables removed from modelling predictors
- accepted binary target column: `target_bad`

In [ ]:
def parse_lending_club_month_year(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, format='%b-%Y', errors='coerce')


def find_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def load_nb02_dataset():
    candidate_files = []
    for nb02_dir in NB02_DIR_CANDIDATES:
        candidate_files.extend([
            nb02_dir / 'initial_modeling_dataset.pkl',
            nb02_dir / 'modeling_dataset_initial.pkl',
            nb02_dir / 'modeling_dataset.pkl',
            nb02_dir / 'df_modeling.pkl',
            nb02_dir / 'target_definition_and_leakage_artifacts.pkl'
        ])

    for path in candidate_files:
        if not path.exists():
            continue
        obj = pd.read_pickle(path)
        if isinstance(obj, pd.DataFrame):
            return obj.copy(), str(path), None
        if isinstance(obj, dict):
            for key in ['df_modeling', 'modeling_dataset', 'initial_modeling_dataset', 'df_model', 'df_clean', 'df_population', 'data', 'df']:
                if key in obj and isinstance(obj[key], pd.DataFrame):
                    return obj[key].copy(), str(path), obj
    return None, None, None


def load_raw_dataset():
    raw_path = find_existing_path(RAW_DATA_CANDIDATES)
    if raw_path is None:
        raise FileNotFoundError('Could not find raw loan_processed_data.csv or Notebook 02 modelling dataset.')
    return pd.read_csv(raw_path), str(raw_path)


df, loaded_from, nb02_artifacts = load_nb02_dataset()

if df is None:
    print('Notebook 02 modelling dataset not found. Loading raw data and applying defensive target/population rules.')
    df, loaded_from = load_raw_dataset()
else:
    print('Loaded Notebook 02 modelling dataset.')

print('Loaded from:', loaded_from)
print('Shape:', df.shape)
df.head()

## 3. Confirm target and modelling population

This notebook uses `target_bad` as the accepted target from Notebook 02.

If the fallback raw dataset is loaded, the target is reconstructed from `loan_status`.

In [ ]:
# Target handling
if TARGET_COL not in df.columns:
    if ORIGINAL_TARGET_COL not in df.columns:
        raise ValueError(
            f"Expected accepted target column '{TARGET_COL}'. It was not found, and raw status column "
            f"'{ORIGINAL_TARGET_COL}' was also not found. Available columns include: {list(df.columns)[:30]}..."
        )
    df[TARGET_COL] = np.where(df[ORIGINAL_TARGET_COL].eq(BAD_LABEL), 1,
                              np.where(df[ORIGINAL_TARGET_COL].eq(GOOD_LABEL), 0, np.nan))
    print(f"Created '{TARGET_COL}' from '{ORIGINAL_TARGET_COL}'.")
else:
    print(f"Using existing accepted target column: '{TARGET_COL}'.")


if 'issue_quarter' not in df.columns and 'issue_d' in df.columns:
    issue_dt = parse_lending_club_month_year(df['issue_d'])
    df['issue_date'] = issue_dt
    df['issue_quarter'] = issue_dt.dt.to_period('Q').astype(str)

if 'issue_quarter' in df.columns:
    before = len(df)
    df = df.loc[~df['issue_quarter'].isin(EXCLUDED_ISSUE_QUARTERS)].copy()
    after = len(df)
    if before != after:
        print(f'Defensive maturity-bias cut applied: removed {before-after:,} rows from {EXCLUDED_ISSUE_QUARTERS}.')

# Keep valid binary target only
df = df.loc[df[TARGET_COL].isin([0, 1])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

TARGET_SUMMARY = pd.DataFrame({
    'metric': ['total_accounts', 'good_count', 'bad_count', 'observed_bad_rate'],
    'value': [
        len(df),
        int((df[TARGET_COL] == 0).sum()),
        int((df[TARGET_COL] == 1).sum()),
        float(df[TARGET_COL].mean())
    ]
})

TARGET_SUMMARY

## 4. Dataset and variable overview

In [ ]:
identifier_like_cols = [
    c for c in df.columns
    if c.lower() in ['id', 'member_id'] or c.lower().endswith('_id')
]

date_like_cols = [
    c for c in df.columns
    if ('date' in c.lower()) or c.endswith('_d') or c in ['issue_quarter', 'issue_year']
]

exclude_from_predictor_review = set([TARGET_COL])

if ORIGINAL_TARGET_COL in df.columns:
    exclude_from_predictor_review.add(ORIGINAL_TARGET_COL)

predictor_cols = [c for c in df.columns if c not in exclude_from_predictor_review]

numeric_cols = df[predictor_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = df[predictor_cols].select_dtypes(exclude=np.number).columns.tolist()

variable_overview = pd.DataFrame({
    'variable': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [float(df[c].isna().mean()) for c in df.columns],
    'unique_count': [int(df[c].nunique(dropna=False)) for c in df.columns],
    'role': [
        'target' if c == TARGET_COL else
        'original_target_source' if c == ORIGINAL_TARGET_COL else
        'identifier_like' if c in identifier_like_cols else
        'date_like' if c in date_like_cols else
        'numeric_candidate' if c in numeric_cols else
        'categorical_candidate' if c in categorical_cols else
        'other'
        for c in df.columns
    ]
})

print('Rows:', len(df))
print('Columns:', df.shape[1])
print('Numeric candidate variables:', len(numeric_cols))
print('Categorical candidate variables:', len(categorical_cols))

variable_overview.head(20)

## 5. Overall target and vintage stability after maturity-bias cut

In [ ]:
if 'issue_quarter' in df.columns:
    vintage_summary = (
        df.groupby('issue_quarter')
          .agg(
              total_accounts=(TARGET_COL, 'size'),
              good_count=(TARGET_COL, lambda x: int((x == 0).sum())),
              bad_count=(TARGET_COL, lambda x: int((x == 1).sum())),
              observed_bad_rate=(TARGET_COL, 'mean')
          )
          .reset_index()
          .sort_values('issue_quarter')
    )
else:
    vintage_summary = pd.DataFrame()

vintage_summary.tail(10)

In [ ]:
if not vintage_summary.empty:
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.bar(vintage_summary['issue_quarter'], vintage_summary['total_accounts'])
    ax1.set_ylabel('Total accounts')
    ax1.set_xlabel('Issue quarter')
    ax1.tick_params(axis='x', rotation=90)
    ax2 = ax1.twinx()
    ax2.plot(vintage_summary['issue_quarter'], vintage_summary['observed_bad_rate'], color='red', marker='o', linewidth=2.5)
    ax2.set_ylabel('Observed bad rate')
    plt.title('Vintage Volume and Observed Bad Rate after Population Cut')
    plt.tight_layout()
    vintage_chart_path = CHART_DIR / 'vintage_volume_and_bad_rate.png'
    plt.savefig(vintage_chart_path, dpi=150, bbox_inches='tight')
    plt.show()
else:
    vintage_chart_path = None

## 6. Numeric variable univariate analysis

For each numeric variable, the notebook creates quantile-based bins and calculates:

- total accounts
- goods
- bads
- observed bad rate
- mean / median / min / max of the variable in the bin

These are diagnostic EDA outputs, not final WOE bins.

In [ ]:
def safe_qcut_bins(series, q=10):
    x = series.copy()
    non_missing = x.dropna()
    if non_missing.nunique() <= 1:
        return pd.Series(['Missing' if pd.isna(v) else 'Single value' for v in x], index=x.index, dtype='object')
    try:
        binned = pd.qcut(non_missing, q=min(q, non_missing.nunique()), duplicates='drop')
    except Exception:
        try:
            binned = pd.cut(non_missing, bins=min(q, non_missing.nunique()), duplicates='drop')
        except Exception:
            return pd.Series(['Missing' if pd.isna(v) else 'Unbinned' for v in x], index=x.index, dtype='object')
    out = pd.Series(index=x.index, dtype='object')
    out.loc[non_missing.index] = binned.astype(str)
    out.loc[x.isna()] = 'Missing'
    return out


def summarize_numeric_variable(data, var, target_col=TARGET_COL, q=10):
    temp = data[[var, target_col]].copy()
    temp['bin'] = safe_qcut_bins(temp[var], q=q)
    summary = (
        temp.groupby('bin', dropna=False)
            .agg(
                total_accounts=(target_col, 'size'),
                good_count=(target_col, lambda x: int((x == 0).sum())),
                bad_count=(target_col, lambda x: int((x == 1).sum())),
                observed_bad_rate=(target_col, 'mean'),
                variable_mean=(var, 'mean'),
                variable_median=(var, 'median'),
                variable_min=(var, 'min'),
                variable_max=(var, 'max')
            )
            .reset_index()
    )
    summary.insert(0, 'variable', var)
    summary['share_of_total'] = summary['total_accounts'] / len(data)
    return summary

numeric_univariate_tables = []
for col in numeric_cols:
    if col == TARGET_COL:
        continue
    try:
        numeric_univariate_tables.append(summarize_numeric_variable(df, col, q=N_NUMERIC_BINS))
    except Exception as exc:
        print(f'Skipped numeric variable {col}: {exc}')

numeric_univariate = pd.concat(numeric_univariate_tables, ignore_index=True) if numeric_univariate_tables else pd.DataFrame()

numeric_variable_summary = (
    numeric_univariate
    .groupby('variable')
    .agg(
        bins=('bin', 'nunique'),
        min_bin_size=('total_accounts', 'min'),
        max_bad_rate=('observed_bad_rate', 'max'),
        min_bad_rate=('observed_bad_rate', 'min'),
        bad_rate_spread=('observed_bad_rate', lambda x: float(x.max() - x.min()))
    )
    .reset_index()
    .sort_values('bad_rate_spread', ascending=False)
    if not numeric_univariate.empty else pd.DataFrame()
)

numeric_variable_summary.head(20)

## 7. Categorical variable univariate analysis

For high-cardinality categorical variables, only the top levels are kept individually and the rest are grouped into `Other`. This is only for EDA readability; final treatment will be decided in feature engineering / WOE analysis.

In [ ]:
def summarize_categorical_variable(data, var, target_col=TARGET_COL, max_levels=50):
    temp = data[[var, target_col]].copy()
    temp[var] = temp[var].astype('object').where(temp[var].notna(), 'Missing')

    top_levels = temp[var].value_counts(dropna=False).head(max_levels).index
    temp['level'] = np.where(temp[var].isin(top_levels), temp[var].astype(str), 'Other')

    summary = (
        temp.groupby('level', dropna=False)
            .agg(
                total_accounts=(target_col, 'size'),
                good_count=(target_col, lambda x: int((x == 0).sum())),
                bad_count=(target_col, lambda x: int((x == 1).sum())),
                observed_bad_rate=(target_col, 'mean')
            )
            .reset_index()
            .sort_values(['total_accounts', 'observed_bad_rate'], ascending=[False, False])
    )
    summary.insert(0, 'variable', var)
    summary['share_of_total'] = summary['total_accounts'] / len(data)
    return summary

categorical_univariate_tables = []
for col in categorical_cols:
    try:
        categorical_univariate_tables.append(summarize_categorical_variable(df, col, max_levels=MAX_CATEGORICAL_LEVELS))
    except Exception as exc:
        print(f'Skipped categorical variable {col}: {exc}')

categorical_univariate = pd.concat(categorical_univariate_tables, ignore_index=True) if categorical_univariate_tables else pd.DataFrame()

categorical_variable_summary = (
    categorical_univariate
    .groupby('variable')
    .agg(
        levels=('level', 'nunique'),
        min_level_size=('total_accounts', 'min'),
        max_bad_rate=('observed_bad_rate', 'max'),
        min_bad_rate=('observed_bad_rate', 'min'),
        bad_rate_spread=('observed_bad_rate', lambda x: float(x.max() - x.min()))
    )
    .reset_index()
    .sort_values('bad_rate_spread', ascending=False)
    if not categorical_univariate.empty else pd.DataFrame()
)

categorical_variable_summary.head(20)

## 8. Key business variables — focused views

In [ ]:
key_business_variables = [
    'grade', 'sub_grade', 'term', 'int_rate', 'loan_amnt', 'installment',
    'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'emp_length', 'home_ownership', 'verification_status', 'purpose',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies'
]

available_key_variables = [c for c in key_business_variables if c in df.columns]
available_key_variables

In [ ]:
key_variable_tables = {}

for var in available_key_variables:
    if var in numeric_cols:
        key_variable_tables[var] = summarize_numeric_variable(df, var, q=N_NUMERIC_BINS)
    elif var in categorical_cols:
        key_variable_tables[var] = summarize_categorical_variable(df, var, max_levels=MAX_CATEGORICAL_LEVELS)

# Display a compact list of key variables ranked by bad-rate spread
key_summary_rows = []
for var, table in key_variable_tables.items():
    level_col = 'bin' if 'bin' in table.columns else 'level'
    key_summary_rows.append({
        'variable': var,
        'type': 'numeric' if var in numeric_cols else 'categorical',
        'groups': table[level_col].nunique(),
        'min_bad_rate': table['observed_bad_rate'].min(),
        'max_bad_rate': table['observed_bad_rate'].max(),
        'bad_rate_spread': table['observed_bad_rate'].max() - table['observed_bad_rate'].min(),
        'min_group_size': table['total_accounts'].min()
    })

key_variable_summary = pd.DataFrame(key_summary_rows).sort_values('bad_rate_spread', ascending=False) if key_summary_rows else pd.DataFrame()
key_variable_summary

## 9. Charts for selected key variables

In [ ]:
def save_bad_rate_chart(table, var, label_col):
    plot_df = table.copy()
    if len(plot_df) > 20:
        plot_df = plot_df.sort_values('total_accounts', ascending=False).head(20)
    plot_df[label_col] = plot_df[label_col].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.bar(plot_df[label_col], plot_df['total_accounts'])
    ax1.set_ylabel('Total accounts')
    ax1.set_xlabel(var)
    ax1.tick_params(axis='x', rotation=45)

    ax2 = ax1.twinx()
    ax2.plot(plot_df[label_col], plot_df['observed_bad_rate'], color='red', marker='o', linewidth=2.5)
    ax2.set_ylabel('Observed bad rate')

    plt.title(f'{var}: Volume and Observed Bad Rate')
    plt.tight_layout()
    safe_var = ''.join(ch if ch.isalnum() or ch in ['_', '-'] else '_' for ch in var)
    path = CHART_DIR / f'{safe_var}_volume_bad_rate.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    return path

selected_chart_variables = [
    v for v in ['grade', 'sub_grade', 'term', 'int_rate', 'dti', 'annual_inc', 'fico_range_high', 'revol_util', 'purpose']
    if v in key_variable_tables
]

chart_paths = {}
for var in selected_chart_variables:
    table = key_variable_tables[var]
    label_col = 'bin' if 'bin' in table.columns else 'level'
    try:
        chart_paths[var] = save_bad_rate_chart(table, var, label_col)
    except Exception as exc:
        print(f'Could not plot {var}: {exc}')

chart_paths

## 10. Missingness and risk relationship

Missing values can be informative in credit risk. This section checks whether records with missing values have materially different observed bad rates.

In [ ]:
missing_risk_rows = []
for col in predictor_cols:
    missing_flag = df[col].isna()
    if missing_flag.sum() == 0:
        continue
    missing_risk_rows.append({
        'variable': col,
        'missing_count': int(missing_flag.sum()),
        'missing_pct': float(missing_flag.mean()),
        'bad_rate_missing': float(df.loc[missing_flag, TARGET_COL].mean()) if missing_flag.any() else np.nan,
        'bad_rate_non_missing': float(df.loc[~missing_flag, TARGET_COL].mean()) if (~missing_flag).any() else np.nan,
        'bad_rate_difference': (
            float(df.loc[missing_flag, TARGET_COL].mean() - df.loc[~missing_flag, TARGET_COL].mean())
            if missing_flag.any() and (~missing_flag).any() else np.nan
        )
    })

missing_risk_summary = pd.DataFrame(missing_risk_rows)
if not missing_risk_summary.empty:
    missing_risk_summary = missing_risk_summary.sort_values('bad_rate_difference', key=lambda s: s.abs(), ascending=False)
missing_risk_summary.head(20)

## 11. Initial EDA observations table

This table is manually/automatically initialized to support later README and presentation materials. It is not a model decision table yet.

In [ ]:
observations = []

def add_observation(topic, observation, implication):
    observations.append({'topic': topic, 'observation': observation, 'implication': implication})

add_observation(
    'Target',
    f"Observed bad rate in the modelling population is {df[TARGET_COL].mean():.2%}.",
    'Class imbalance appears moderate and suitable for baseline logistic regression without mandatory resampling.'
)

if not vintage_summary.empty:
    add_observation(
        'Vintage stability',
        'Notebook 02 excluded 2018Q3 and 2018Q4 due to likely maturity bias / insufficient seasoning.',
        'EDA and univariate analysis are performed on the seasoned modelling population.'
    )

if not key_variable_summary.empty:
    top_vars = ', '.join(key_variable_summary.head(5)['variable'].tolist())
    add_observation(
        'Univariate risk drivers',
        f'Top candidate variables by univariate bad-rate spread include: {top_vars}.',
        'These variables should be prioritized for WOE/IV review, monotonicity checks and correlation analysis.'
    )

if not missing_risk_summary.empty:
    top_missing_var = missing_risk_summary.iloc[0]['variable']
    add_observation(
        'Missingness',
        f'Missingness appears most differentiated for {top_missing_var}.',
        'Missing indicators or separate missing bins may be useful in later feature engineering / WOE binning.'
    )

eda_observations = pd.DataFrame(observations)
eda_observations

## 12. Save artifacts

The notebook saves:

- Excel report with target, vintage, variable overview, numeric/categorical univariate outputs and key observations
- Pickle artifact dictionary
- EDA modelling dataset as `.pkl`
- charts under the `charts/` folder

In [ ]:
artifacts = {
    'metadata': {
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'notebook': NOTEBOOK_NAME,
        'loaded_from': loaded_from,
        'target_col': TARGET_COL,
        'excluded_issue_quarters': EXCLUDED_ISSUE_QUARTERS,
        'notes': 'Observed bad rate is empirical bad_count / total_accounts, not modelled PD.'
    },
    'target_summary': TARGET_SUMMARY,
    'variable_overview': variable_overview,
    'vintage_summary': vintage_summary,
    'numeric_variable_summary': numeric_variable_summary,
    'numeric_univariate': numeric_univariate,
    'categorical_variable_summary': categorical_variable_summary,
    'categorical_univariate': categorical_univariate,
    'key_variable_summary': key_variable_summary,
    'key_variable_tables': key_variable_tables,
    'missing_risk_summary': missing_risk_summary,
    'eda_observations': eda_observations,
    'chart_paths': {k: str(v) for k, v in chart_paths.items()}
}

with open(PKL_OUTPUT_PATH, 'wb') as f:
    pickle.dump(artifacts, f)

df.to_pickle(EDA_DATASET_PATH)

with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine='openpyxl') as writer:
    TARGET_SUMMARY.to_excel(writer, sheet_name='target_summary', index=False)
    variable_overview.to_excel(writer, sheet_name='variable_overview', index=False)
    if not vintage_summary.empty:
        vintage_summary.to_excel(writer, sheet_name='vintage_summary', index=False)
    if not numeric_variable_summary.empty:
        numeric_variable_summary.to_excel(writer, sheet_name='numeric_summary', index=False)
    if not numeric_univariate.empty:
        numeric_univariate.to_excel(writer, sheet_name='numeric_univariate', index=False)
    if not categorical_variable_summary.empty:
        categorical_variable_summary.to_excel(writer, sheet_name='categorical_summary', index=False)
    if not categorical_univariate.empty:
        categorical_univariate.to_excel(writer, sheet_name='categorical_univariate', index=False)
    if not key_variable_summary.empty:
        key_variable_summary.to_excel(writer, sheet_name='key_variable_summary', index=False)
    if not missing_risk_summary.empty:
        missing_risk_summary.to_excel(writer, sheet_name='missing_risk', index=False)
    eda_observations.to_excel(writer, sheet_name='eda_observations', index=False)

print('Saved Excel report:', EXCEL_OUTPUT_PATH.resolve())
print('Saved pickle artifacts:', PKL_OUTPUT_PATH.resolve())
print('Saved EDA dataset:', EDA_DATASET_PATH.resolve())
print('Saved charts directory:', CHART_DIR.resolve())

## 13. Conclusion

This notebook provides broad EDA and univariate risk analysis on the accepted modelling population.

- use `target_bad` as the accepted binary target
- continue to treat `observed_bad_rate` as empirical historical outcome rate, not modelled PD
- use the matured modelling population after excluding `2018Q3` and `2018Q4`
- review top univariate risk drivers for feature engineering and WOE/IV binning
- investigate missingness patterns before deciding final imputation/binning strategy